# Notebook 03: Architectural Diversity & Centered Kernel Alignment (CKA)
### MOSAIC-Med Research Study: Homogeneous vs Heterogeneous Expert Populations

---

### Hypothesis 2 & Parameter-Matched Control
Does combining models with **distinct inductive biases** (heterogeneous architectures) produce complementary feature representations that surpass homogeneous ensembles of identical copies?

We compare two 4-expert populations under an invariant ~2.0M parameter budget (+0.92% matching margin):
1. **Homogeneous Population (4$\times$ResNet)**: $1,975,572$ parameters (126.8 MFLOPs)
2. **Heterogeneous Population**: $1,993,688$ parameters (413.7 MFLOPs)
   - **ResNet-1D** (493,893 params, 31.69 MFLOPs) — Deep residual temporal convolutions
   - **MultiScale-CNN** (508,909 params, 50.72 MFLOPs) — Parallel multi-resolution temporal kernels (5, 11, 23 samples)
   - **TCN-1D** (490,661 params, 245.86 MFLOPs) — Exponentially dilated causal convolutions ($d \in \{1, 2, 4, 8\}$)
   - **ConvAttention-1D** (500,225 params, 85.40 MFLOPs) — Convolutional stem + 2-layer self-attention Transformer

Representation diversity is quantified using Linear **Centered Kernel Alignment (CKA)** on penultimate activations over the held-out test cohort.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
from IPython.display import Image, display

print("Modules loaded.")


## 1. Parameter-Matched Statistical Comparison (Seeds 42–46)


In [ ]:
matched_df = pd.read_csv('results/tables/parameter_matched_control.csv')
print("Parameter-Matched Control Results (5 Seeds, df=4):")
matched_df[['Metric', 'Homogeneous 4xResNet (Mean ± SD)', 'Heterogeneous Matched (Mean ± SD)', 'Delta (Hetero - Homo)', 'Raw p-value', 'Holm-adjusted p-value', 'Statistically Significant (α=0.05)']]


## 2. Centered Kernel Alignment (CKA) Representation Analysis
A lower CKA score between two models indicates higher feature orthogonality.


In [ ]:
cka_df = pd.read_csv('results/tables/cka_comparison.csv')
print("Mean Pairwise Linear CKA Scores:")
print(f"Homogeneous 4xResNet Mean CKA:  {cka_df[cka_df['population'] == 'Homogeneous']['mean_cka'].values[0]:.4f}")
print(f"Heterogeneous Matched Mean CKA: {cka_df[cka_df['population'] == 'Heterogeneous']['mean_cka'].values[0]:.4f}")

display(Image('results/images/cka_representation_diversity.png'))


## 3. Scientific Findings & Statistical Rigor
- **Observed Trend (H2)**: The heterogeneous ensemble achieves higher discrimination (AUROC $0.9110 \pm 0.0006$ vs $0.9084 \pm 0.0008$) and calibration (Brier $0.0867 \pm 0.0005$ vs $0.0881 \pm 0.0002$).
- **Statistical Significance Status**: Across 5 seeds, paired $t$-test yields raw $p = 0.0164$, which adjusts to **Holm $p = 0.0577$** across the $m=4$ metric family (AUROC, AUPRC, Brier, ECE). Because $p > 0.05$, this result represents a **consistent descriptive improvement trend**, but does not achieve strict significance at the conventional $\alpha = 0.05$ threshold.
- **CKA Diversity**: Heterogeneous models achieve lower representational similarity ($0.9008$ vs $0.9458$), indicating orthogonal feature spaces. However, CKA characterizes representation structure and should not be treated as a causal proof of performance.
